# Auditer un formulaire conditionnel — l'état que vous ne voyez pas

*Recycler la documentation du projet LivresAgités (en sommeil) vers le dépôt
pédagogique. Ce notebook illustre la feature **AI Forms** d'AI-Engine — l'une
des deux fonctionnalités GenAI cœur du dossier qui n'avait, jusqu'ici, ni
section de parcours ni notebook compagnon (l'autre étant le Copilot
Gutenberg, Parcours 1).*

> **Thèse.** Un formulaire à logique conditionnelle — dont les champs
> apparaissent ou disparaissent selon les réponses antérieures — est une
> **machine à états implicite**. Le schéma se lit comme une simple liste de
> champs, mais le comportement réel est *émergent* : quels chemins existent,
> lesquels terminent, lesquels déclenchent un appel au LLM, ne se déduisent
> pas de la lecture du schéma. Ils s'énumèrent. Et leur nombre peut être
> **exponentiel** dans le nombre de conditions.

Ce notebook ne dépend d'aucun service : pas de réseau, pas de modèle, pas de
clé. Le formulaire et son moteur d'énumération sont construits en mémoire,
sur le fixture synthétique « Maison Valmont ». Ce qui est enseigné est la
*structure* du comportement d'un formulaire conditionnel, pas une mesure sur
une instance réelle.


## 1. AI Forms en une phrase

**AI Forms** est le module d'AI-Engine qui pousse les formulaires WordPress
au-delà de la simple collecte : chaque soumission peut être **traitée par un
LLM** (synthèse, extraction, classification, traduction) avant stockage ou
notification. La pièce maîtresse, pédagogiquement, n'est pas le LLM — c'est
la **logique conditionnelle** : la visibilité d'un champ dépend des réponses
données aux champs précédents. Un même formulaire présente donc des « pages »
différentes selon qui le remplit et comment.

C'est cette logique conditionnelle qui transforme un formulaire en machine à
états. Et comme toute machine à états, son comportement effectif (les
chemins atteignables) ne se lit pas sur la définition statique : il se
calcule.


## 2. Le fixture — un formulaire de soumission de manuscrit

On monte la « Maison Valmont » : un formulaire de dépôt de manuscrit à
**sept champs**, dont la visibilité est conditionnelle. Les conditions sont
représentées en **données** (des tuples), pas en code (pas de `lambda`) —
c'est plus long à écrire, mais cela rend le formulaire **inspectable** : on
peut lire une condition comme on lit une donnée, sans exécuter de fonction.

Chaque champ porte :
- une **condition** (tuple) — predicat sur les réponses antérieures ;
- des **valeurs** possibles (l'énumération se branche sur chacune) ;
- une **action** — `None` (rien) ou un identifiant d'action LLM.


In [1]:
# Aucune dependance externe. Tout est en memoire.

# --- Formulaire de soumission Valmont (7 champs conditionnels) ---
# Condition = un tuple (operateur, champ_anterieur, valeur)
#   ("true",)                  -> toujours vrai
#   ("eq",  champ, val)        -> reponses[champ] == val
#   ("in",  champ, (v1, v2))   -> reponses[champ] dans le tuple
#   ("gt",  champ, val)        -> reponses.get(champ, 0) > val
#   ("lt",  champ, val)        -> reponses.get(champ, 0) < val
formulaire = {
    "courriel":     {"condition": ("true",),                  "valeurs": ["auteur@valmont.org"], "action": None},
    "genre":        {"condition": ("true",),                  "valeurs": ["poesie", "prose", "essai"], "action": None},
    "nb_pages":     {"condition": ("in", "genre", ("prose", "essai")), "valeurs": [40, 120, 250], "action": None},
    "resume_long":  {"condition": ("gt", "nb_pages", 200),     "valeurs": ["resume_a", "resume_b"], "action": "llm_synthese"},
    "deja_edite":   {"condition": ("eq", "genre", "prose"),    "valeurs": [True, False], "action": None},
    "nom_editeur":  {"condition": ("eq", "deja_edite", True),  "valeurs": ["editeur_x"], "action": "llm_verification"},
    "note_lecture": {"condition": ("lt", "nb_pages", 0),       "valeurs": [1, 2, 3], "action": None},
}

print("Formulaire de " + str(len(formulaire)) + " champs.")
for nom, c in formulaire.items():
    act = c["action"] if c["action"] else "-"
    print("  " + nom + " | cond=" + str(c["condition"]) + " | " + str(len(c["valeurs"])) + " valeurs | action=" + str(act))


Formulaire de 7 champs.
  courriel | cond=('true',) | 1 valeurs | action=-
  genre | cond=('true',) | 3 valeurs | action=-
  nb_pages | cond=('in', 'genre', ('prose', 'essai')) | 3 valeurs | action=-
  resume_long | cond=('gt', 'nb_pages', 200) | 2 valeurs | action=llm_synthese
  deja_edite | cond=('eq', 'genre', 'prose') | 2 valeurs | action=-
  nom_editeur | cond=('eq', 'deja_edite', True) | 1 valeurs | action=llm_verification
  note_lecture | cond=('lt', 'nb_pages', 0) | 3 valeurs | action=-


### Lire le schéma

Sept champs, à l'œil un formulaire modeste. Déjà, à la lecture, on devine
que `resume_long` n'apparaît que pour les manuscrits de plus de 200 pages,
et que `nom_editeur` ne sort que pour la prose déjà éditée. Mais **combien
de chemins distincts** ce formulaire génère-t-il ? La lecture ne le dit pas.
C'est la question que ce notebook pose — et résout par énumération.


## 3. Le moteur — évaluer une condition, énumérer les chemins

Deux fonctions suffisent. La première **évalue** une condition (le tuple)
contre un ensemble de réponses. La seconde **énumère** tous les chemins
terminaux : pour chaque champ, si sa condition est satisfaite on se
ramifie sur chaque valeur possible, sinon on passe (champ masqué).


In [2]:
def condition_satisfaite(cond, reponses):
    """Evalue un tuple-condition contre les reponses anterieures."""
    op = cond[0]
    if op == "true":
        return True
    champ, val = cond[1], cond[2]
    if op == "eq":
        return reponses.get(champ) == val
    if op == "ne":
        return reponses.get(champ) != val
    if op == "gt":
        return reponses.get(champ, 0) > val
    if op == "lt":
        return reponses.get(champ, 0) < val
    if op == "in":
        return reponses.get(champ) in val
    return False


# Sanity check : la condition "nb_pages > 200" sur un manuscrit de 40 pages.
print("nb_pages=40  > 200 ? " + str(condition_satisfaite(("gt", "nb_pages", 200), {"nb_pages": 40})))
print("nb_pages=250 > 200 ? " + str(condition_satisfaite(("gt", "nb_pages", 200), {"nb_pages": 250})))
print("genre=prose in (prose,essai) ? " + str(condition_satisfaite(("in", "genre", ("prose", "essai")), {"genre": "prose"})))
print("genre=poesie in (prose,essai) ? " + str(condition_satisfaite(("in", "genre", ("prose", "essai")), {"genre": "poesie"})))


nb_pages=40  > 200 ? False
nb_pages=250 > 200 ? True
genre=prose in (prose,essai) ? True
genre=poesie in (prose,essai) ? False


In [3]:
def enumerer_chemins(formulaire):
    """Enumere tous les chemins terminaux atteignables du formulaire.

    Un chemin = un dict d'affectations des champs VISIBLES sur ce chemin.
    Pour chaque champ : si sa condition est satisfaite, on se ramifie sur
    chacune de ses valeurs ; sinon le champ est masque et on continue.
    """
    chemins = [{}]
    for nom, champ in formulaire.items():
        suivants = []
        for reponses in chemins:
            if condition_satisfaite(champ["condition"], reponses):
                for valeur in champ["valeurs"]:
                    r2 = dict(reponses)
                    r2[nom] = valeur
                    suivants.append(r2)
            else:
                suivants.append(reponses)
        chemins = suivants
    return chemins


chemins = enumerer_chemins(formulaire)
print("Chemins terminaux atteignables : " + str(len(chemins)))


Chemins terminaux atteignables : 13


## 4. L'explosion — combien de chemins pour sept champs ?

Sept champs, et pourtant le formulaire n'engendre pas sept états. Il en
engendre bien plus, parce que chaque condition crée un point de branchement.
Mesurons.


In [4]:
from collections import Counter

print("Nombre de chemins terminaux : " + str(len(chemins)))
print()

# Distribution des longueurs (nombre de champs visibles par chemin).
longueurs = Counter(len(c) for c in chemins)
print("Distribution des longueurs (champs visibles par chemin) :")
for k in sorted(longueurs):
    print("  " + str(k) + " champs : " + str(longueurs[k]) + " chemin(s)")
print()

# A titre de comparaison : le produit cartesien brut de toutes les valeurs.
import math
produit_brut = 1
for champ in formulaire.values():
    produit_brut *= len(champ["valeurs"])
print("Produit cartesien brut (si AUCUNE condition) : " + str(produit_brut))
print("Avec conditions : " + str(len(chemins)) + " -> " + str(round(100 * len(chemins) / produit_brut)) + "% du brut.")


Nombre de chemins terminaux : 13

Distribution des longueurs (champs visibles par chemin) :
  2 champs : 1 chemin(s)
  3 champs : 2 chemin(s)
  4 champs : 4 chemin(s)
  5 champs : 4 chemin(s)
  6 champs : 2 chemin(s)

Produit cartesien brut (si AUCUNE condition) : 108
Avec conditions : 13 -> 12% du brut.


### Lecture

Sept champs engendrent **13 chemins terminaux atteignables** (contre un
produit cartésien brut de 108 si aucune condition ne masquait jamais rien).
Le conditionnel réduit l'espace — c'est sa raison d'être — mais il le rend
surtout **non lisible** : aucune des 13 combinaisons n'est visible sur le
schéma. On lit « sept champs » ; l'utilisateur rencontre l'un des **treize
formulaires différents** que ce dispositif produit réellement.

C'est le premier enseignement : la taille apparente du formulaire (le
nombre de champs déclarés) ne dit rien de sa taille effective (le nombre
d'états distincts). Pour un petit formulaire c'est une curiosité ; pour un
formulaire administratif de cinquante champs conditionnels, c'est une
explosion combinatoire que personne n'audite.


## 5. Les chemins qui invoquent le LLM — le coût caché

AI Forms peut déclencher une action LLM à la soumission. Ici, deux champs
portent une action : `resume_long` (synthèse) et `nom_editeur` (vérification
de l'éditeur). Mais **un champ ne s'exécute que s'il est visible** — donc
l'action LLM ne se déclenche que sur les chemins où le champ apparaît. Le
coût en appels LLM est, lui aussi, émergent.


In [5]:
def cout_llm(chemin, formulaire):
    """Nombre d'appels LLM declenches par un chemin = nombre de champs
    visibles portant une action non-None."""
    return sum(1 for nom in chemin if formulaire[nom]["action"] is not None)


coûts = Counter(cout_llm(c, formulaire) for c in chemins)
print("Répartition des chemins par nombre d'appels LLM :")
for k in sorted(coûts):
    print("  " + str(k) + " appel(s) LLM : " + str(coûts[k]) + " chemin(s)")
print()

nb_avec_llm = sum(1 for c in chemins if cout_llm(c, formulaire) > 0)
print("Chemins invoquant AU MOINS un appel LLM : " + str(nb_avec_llm) + " / " + str(len(chemins)))
print("  -> " + str(round(100 * nb_avec_llm / len(chemins))) + " % des chemins coûtent un appel LLM.")
print()

# Detail des chemins a 2 appels (les plus couteux).
double = [c for c in chemins if cout_llm(c, formulaire) == 2]
print("Chemins à double appel LLM (" + str(len(double)) + ") :")
for c in double:
    print("  " + str(c))


Répartition des chemins par nombre d'appels LLM :
  0 appel(s) LLM : 5 chemin(s)
  1 appel(s) LLM : 6 chemin(s)
  2 appel(s) LLM : 2 chemin(s)

Chemins invoquant AU MOINS un appel LLM : 8 / 13
  -> 62 % des chemins coûtent un appel LLM.

Chemins à double appel LLM (2) :
  {'courriel': 'auteur@valmont.org', 'genre': 'prose', 'nb_pages': 250, 'resume_long': 'resume_a', 'deja_edite': True, 'nom_editeur': 'editeur_x'}
  {'courriel': 'auteur@valmont.org', 'genre': 'prose', 'nb_pages': 250, 'resume_long': 'resume_b', 'deja_edite': True, 'nom_editeur': 'editeur_x'}


### Lecture

Sur 13 chemins, **8 déclenchent au moins un appel LLM — près des deux
tiers**. Et **2 chemins en déclenchent deux** (synthèse *et* vérification :
la prose de plus de 200 pages, déjà éditée). Lu sur le schéma, on voit
« deux champs LLM ». Mesuré sur les chemins, on découvre que **le formulaire
le plus coûteux paie deux appels là où le moins coûteux n'en paie aucun** —
pour un écart que rien, dans la définition statique, ne signalait.

C'est le second enseignement : le coût d'exploitation d'un formulaire AI
Forms est une propriété des chemins, pas des champs. Un audit qui se borne à
lister les champs à action LLM sous-estime le coût réel (il ignore les
chemins à appels multiples) et le surévalue (il compte un champ qui n'est
visible sur aucun chemin).


## 6. Les champs morts — ce que personne ne verra jamais

Dernière catégorie, la plus sournoise : un champ peut être **déclaré mais
jamais visible**, parce que sa condition n'est jamais satisfaite quel que
soit le chemin. Le champ `note_lecture` (condition `nb_pages < 0`) en est
l'exemple : aucun manuscrit n'a un nombre de pages négatif, donc le champ
n'apparaît jamais. Il vit dans le schéma, pas dans le formulaire effectif.


In [6]:
# Verifions a la main : note_lecture apparait-il dans un seul chemin ?
present = any("note_lecture" in c for c in chemins)
print("note_lecture present dans au moins un chemin : " + str(present))
print()

# Plus generalement : pour chaque champ, dans combien de chemins apparait-il ?
print("Visibilite de chaque champ :")
for nom in formulaire:
    nb = sum(1 for c in chemins if nom in c)
    marqueur = "  <-- CHAMP MORT" if nb == 0 else ""
    print("  " + nom + " : " + str(nb) + " / " + str(len(chemins)) + " chemins" + marqueur)


note_lecture present dans au moins un chemin : False

Visibilite de chaque champ :
  courriel : 13 / 13 chemins
  genre : 13 / 13 chemins
  nb_pages : 12 / 13 chemins
  resume_long : 6 / 13 chemins
  deja_edite : 8 / 13 chemins
  nom_editeur : 4 / 13 chemins
  note_lecture : 0 / 13 chemins  <-- CHAMP MORT


### Lecture

`note_lecture` apparaît dans **0 chemin sur 12** : c'est un champ mort. Il
ne consomme rien (pas d'action LLM), mais il **trompe l'audit** : un lecteur
du schéma le comptera comme une fonctionnalité live, alors qu'il est
inaccessible. Dans un formulaire administratif réel, les champs morts
s'accumulent avec les évolutions — une condition durcie ici, un champ
renommé là — et transforment le schéma en palimpseste où l'effective et
l'inerte cohabitent sans distinction.

C'est le troisième enseignement : **le schéma n'est pas le formulaire**.
Trois grandeurs mesurées ce notebook (chemins, coût LLM, champs morts) sont
toutes des propriétés *émergentes* qu'aucune lecture statique ne donne.


## 7. Provenance et limites

**Ce que ce notebook mesure.** La *structure du comportement* d'un
formulaire conditionnel : étant donné un schéma (champs + conditions +
actions), combien d'états terminaux il produit, combien coûtent ces états
en appels LLM, et quels champs déclarés ne servent jamais. Tout est
déterministe sur fixture synthétique.

**Ce qu'il ne mesure pas.** L'ergonomie réelle du formulaire vu par
l'utilisateur (un chemin peut être rare mais jamais emprunté en pratique),
la latence des appels LLM, les validations croisées entre champs. La
représentation des conditions par tuples est volontairement simpliste — un
vrai moteur de conditions supporte des expressions booléennes imbriquées ;
le principe (énumérer pour auditer) est identique.

**La limite du procédé.** L'énumération exhaustive a un coût : un
formulaire de $N$ champs conditionnels peut produire un nombre de chemins
exponentiel en $N$. Auditer en énumérant devient prohibitif sur les gros
formulaires — c'est précisément pourquoi cette auditabilité est rarement
faite, et pourquoi les champs morts et les chemins coûteux s'y cachent.

**Pour aller plus loin.**
- [`auditer-un-serveur-mcp.ipynb`](auditer-un-serveur-mcp.ipynb) — même
  esprit d'audit (mesurer une propriété non-lisible sur la définition
  statique) appliqué à un catalogue d'outils MCP.
- [`consommer-vs-exposer-le-mcp.ipynb`](consommer-vs-exposer-le-mcp.ipynb) —
  l'autre face MCP : comparer deux catalogues.
- [`livresagites-parcours.md`](livresagites-parcours.md) Parcours 4 — la
  prose dont ce notebook est l'illustration exécutable.


## 8. Exercices

Les trois exercices suivants manipulent le formulaire synthétique et son
moteur (`enumerer_chemins`, `condition_satisfaite`). Les stub sont à
compléter — `return None` ou `pass`.


### Exercice 1 — distribution des longueurs

Écrire une fonction `compter_par_longueur(formulaire)` qui renvoie un dict
`{longueur: nombre_de_chemins}` — pour chaque longueur possible (nombre de
champs visibles), combien de chemins terminaux ont exactement cette
longueur.


In [7]:
def compter_par_longueur(formulaire):
    """Renvoie {longueur: nb_chemins} -- distribution du nombre de champs
    visibles par chemin terminal.
    """
    # TODO : utiliser enumerer_chemins() et regrouper par len(chemin).
    return None


### Exercice 2 — regrouper les chemins par coût LLM

Écrire une fonction `regrouper_par_cout(formulaire)` qui renvoie un dict
`{cout: [chemins]}` — les chemins terminaux groupés par nombre d'appels LLM
(0, 1, 2…). Permet d'isoler les chemins les plus coûteux.


In [8]:
def regrouper_par_cout(formulaire):
    """Renvoie {cout_llm: [chemins]} -- chemins terminaux groupes par
    nombre d'appels LLM qu'ils declenchent.
    """
    # TODO : utiliser enumerer_chemins() et cout_llm().
    return None


### Exercice 3 — détecter les champs morts

Écrire une fonction `champs_jamais_visibles(formulaire)` qui renvoie la
liste des champs déclarés mais absents de **tous** les chemins terminaux
(leurs conditions ne sont jamais satisfaites). `note_lecture` doit y
figurer.


In [9]:
def champs_jamais_visibles(formulaire):
    """Renvoie la liste des champs declares mais presents dans aucun
    chemin terminal atteignable.
    """
    # TODO : enumerer les chemins, puis chercher les champs absents de tous.
    return None


## 9. Ce que ce notebook enseigne, en une ligne

> **Un formulaire conditionnel n'est pas une liste de champs, c'est un graphe
> d'états.** Le lire comme une liste fait rater trois choses : combien
> d'états existent, combien ils coûtent, et lesquels sont morts. Trois
> réponses qui ne viennent que de l'énumération.
